# 9_Construccion_Golden_Fact_Ventas_v1

**Objetivo:** construir y persistir la tabla de hechos `golden.fact_ventas` a partir de `silver.ventas`, relacionándola con las cuatro dimensiones Golden.

**Granularidad:** una fila por `venta_id`.

**Importante:** el notebook no modifica Bronze, Silver ni las dimensiones ya construidas. Antes de escribir, identifica ventas que no puedan relacionarse con alguna dimensión. Esas filas se conservan separadas en `golden.fact_ventas_no_mapeadas`, para evitar pérdidas silenciosas.

La utilidad bruta se presentará como **estimada**, porque utiliza el costo unitario de referencia de `dim_producto`, que no necesariamente representa el costo histórico de la fecha de venta.

Ejecutaremos las secciones celda por celda y validaremos cada resultado antes de avanzar.

## 1. Importaciones

**¿Qué realiza esta sección?** Importa las funciones de PySpark necesarias para las uniones, transformaciones, cálculos, validaciones y persistencia Delta.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

## 2. Parámetros

**¿Qué realiza esta sección?** Define el catálogo, los esquemas de origen y destino, y los nombres completos de las tablas que utilizaremos.

Los valores predeterminados corresponden al entorno del proyecto. La capa de destino se denomina `golden`, según el documento del proyecto final.

In [ ]:
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")

catalogo = dbutils.widgets.get("catalogo")
containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")

esquemaSilver = "silver"
esquemaGolden = "golden"

ruta_golden = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/golden/"

tabla_ventas_silver = f"{catalogo}.{esquemaSilver}.ventas"

tabla_dim_fecha = f"{catalogo}.{esquemaGolden}.dim_fecha"
tabla_dim_cliente = f"{catalogo}.{esquemaGolden}.dim_cliente"
tabla_dim_producto = f"{catalogo}.{esquemaGolden}.dim_producto"
tabla_dim_almacen = f"{catalogo}.{esquemaGolden}.dim_almacen"

tabla_fact_ventas = f"{catalogo}.{esquemaGolden}.fact_ventas"
tabla_fact_ventas_no_mapeadas = f"{catalogo}.{esquemaGolden}.fact_ventas_no_mapeadas"

print("Catálogo:", catalogo)
print("Origen:", tabla_ventas_silver)
print("Destino:", f"{catalogo}.{esquemaGolden}")
print("Ruta Golden:", ruta_golden)

## 3. Lectura desde Silver y dimensiones Golden

**¿Qué realiza esta sección?** Lee la tabla depurada de ventas y las cuatro dimensiones persistidas. No vuelve a construir dimensiones ni modifica Silver.

Se muestran los conteos para confirmar que estamos leyendo las tablas esperadas.

In [ ]:
df_ventas_silver = spark.table(tabla_ventas_silver)

df_dim_fecha = spark.table(tabla_dim_fecha)
df_dim_cliente = spark.table(tabla_dim_cliente)
df_dim_producto = spark.table(tabla_dim_producto)
df_dim_almacen = spark.table(tabla_dim_almacen)

print("Ventas Silver:", df_ventas_silver.count())
print("dim_fecha:", df_dim_fecha.count())
print("dim_cliente:", df_dim_cliente.count())
print("dim_producto:", df_dim_producto.count())
print("dim_almacen:", df_dim_almacen.count())

df_ventas_silver.printSchema()

## 4. Diagnóstico de ventas Silver

**¿Qué realiza esta sección?** Revisa el volumen de ventas, la unicidad de `venta_id` y los campos esenciales para construir el hecho.

La capa Silver ya rechazó las ventas con `fecha_venta` nula y duplicados excedentes. Esta sección vuelve a comprobar esas condiciones antes de construir Golden.

In [ ]:
conteo_ventas_silver = df_ventas_silver.count()

print("Ventas Silver:", conteo_ventas_silver)
print("venta_id nulo:", df_ventas_silver.filter(F.col("venta_id").isNull()).count())
print("venta_id duplicado:",
      df_ventas_silver.groupBy("venta_id").count().filter(F.col("count") > 1).count())
print("fecha_venta nula:", df_ventas_silver.filter(F.col("fecha_venta").isNull()).count())
print("cliente_id nulo:", df_ventas_silver.filter(F.col("cliente_id").isNull()).count())
print("producto_id nulo:", df_ventas_silver.filter(F.col("producto_id").isNull()).count())
print("almacen_id nulo:", df_ventas_silver.filter(F.col("almacen_id").isNull()).count())
print("cantidad nula:", df_ventas_silver.filter(F.col("cantidad").isNull()).count())
print("precio_unitario nulo:", df_ventas_silver.filter(F.col("precio_unitario").isNull()).count())
print("descuento_pct nulo:", df_ventas_silver.filter(F.col("descuento_pct").isNull()).count())

assert df_ventas_silver.filter(F.col("venta_id").isNull()).count() == 0, "Hay venta_id nulos."
assert df_ventas_silver.groupBy("venta_id").count().filter(F.col("count") > 1).count() == 0, "Hay venta_id duplicados."
assert df_ventas_silver.filter(F.col("fecha_venta").isNull()).count() == 0, "Hay fechas de venta nulas."
assert df_ventas_silver.filter(F.col("descuento_pct").isNull()).count() == 0, "Hay descuentos nulos en Silver."

## 5. Diagnóstico de integridad con dimensiones Golden

**¿Qué realiza esta sección?** Comprueba que cada venta pueda encontrar su correspondiente fila en `dim_fecha`, `dim_cliente`, `dim_producto` y `dim_almacen`.

Esta comprobación es especialmente importante para productos: cinco productos fueron rechazados de `silver.productos` por tener `costo_unitario` nulo. Una venta puede tener un producto existente en Bronze y Silver de ventas, pero no disponer de una fila correspondiente en `dim_producto`.

La sección identifica esas ventas antes de decidir su tratamiento. No las elimina silenciosamente.

In [ ]:
dim_fecha_lookup = df_dim_fecha.select(
    F.col("fecha").alias("fecha_dim"),
    "fecha_key"
)

dim_cliente_lookup = df_dim_cliente.select(
    F.col("cliente_id").alias("cliente_id_dim"),
    "cliente_key"
)

dim_producto_lookup = df_dim_producto.select(
    F.col("producto_id").alias("producto_id_dim"),
    "producto_key",
    F.col("costo_unitario").alias("costo_unitario_referencia")
)

dim_almacen_lookup = df_dim_almacen.select(
    F.col("almacen_id").alias("almacen_id_dim"),
    "almacen_key"
)

df_ventas_mapeo = (
    df_ventas_silver
    .withColumn("fecha_venta_dia", F.to_date("fecha_venta"))
    .join(dim_fecha_lookup,
          F.col("fecha_venta_dia") == F.col("fecha_dim"), "left")
    .join(dim_cliente_lookup,
          F.col("cliente_id") == F.col("cliente_id_dim"), "left")
    .join(dim_producto_lookup,
          F.col("producto_id") == F.col("producto_id_dim"), "left")
    .join(dim_almacen_lookup,
          F.col("almacen_id") == F.col("almacen_id_dim"), "left")
)

resumen_mapeo = [
    ("Sin fecha_key", df_ventas_mapeo.filter(F.col("fecha_key").isNull()).count()),
    ("Sin cliente_key", df_ventas_mapeo.filter(F.col("cliente_key").isNull()).count()),
    ("Sin producto_key", df_ventas_mapeo.filter(F.col("producto_key").isNull()).count()),
    ("Sin almacen_key", df_ventas_mapeo.filter(F.col("almacen_key").isNull()).count())
]

for nombre, cantidad in resumen_mapeo:
    print(f"{nombre}: {cantidad}")

display(
    df_ventas_mapeo
    .filter(
        F.col("fecha_key").isNull() |
        F.col("cliente_key").isNull() |
        F.col("producto_key").isNull() |
        F.col("almacen_key").isNull()
    )
    .select(
        "venta_id", "fecha_venta", "cliente_id", "producto_id", "almacen_id",
        "fecha_key", "cliente_key", "producto_key", "almacen_key"
    )
    .limit(30)
)

## 6. Construcción de `fact_ventas` y ventas no mapeadas

**¿Qué realiza esta sección?** Separa las ventas que encontraron todas sus claves dimensionales de aquellas que no pudieron relacionarse con alguna dimensión.

Las ventas completamente mapeadas conforman `df_fact_ventas`. Las ventas sin una o más claves se conservan en `df_fact_ventas_no_mapeadas`, con el motivo correspondiente.

Se calculan:
- `importe_bruto = cantidad × precio_unitario`
- `importe_descuento = importe_bruto × descuento_pct / 100`
- `importe_neto = importe_bruto − importe_descuento`
- `costo_estimado = cantidad × costo_unitario_referencia`
- `utilidad_bruta_estimada = importe_neto − costo_estimado`

El costo procede de la dimensión de producto y es un costo de referencia, no necesariamente histórico.

In [ ]:
df_ventas_mapeo = df_ventas_mapeo.withColumn(
    "motivo_no_mapeo",
    F.concat_ws(
        "|",
        F.when(F.col("fecha_key").isNull(), F.lit("FECHA_SIN_DIMENSION")),
        F.when(F.col("cliente_key").isNull(), F.lit("CLIENTE_SIN_DIMENSION")),
        F.when(F.col("producto_key").isNull(), F.lit("PRODUCTO_SIN_DIMENSION")),
        F.when(F.col("almacen_key").isNull(), F.lit("ALMACEN_SIN_DIMENSION"))
    )
)

df_ventas_no_mapeadas = (
    df_ventas_mapeo
    .filter(F.col("motivo_no_mapeo") != "")
    .withColumn("fecha_proceso", F.current_timestamp())
    .select(
        "venta_id", "fecha_venta", "cliente_id", "producto_id", "almacen_id",
        "cantidad", "precio_unitario", "descuento_pct", "metodo_pago",
        "_source_file", "_ingestion_timestamp",
        "motivo_no_mapeo", "fecha_proceso"
    )
)

df_ventas_mapeadas = df_ventas_mapeo.filter(
    F.col("motivo_no_mapeo") == ""
)

df_fact_ventas = (
    df_ventas_mapeadas
    .withColumn(
        "importe_bruto",
        (F.col("cantidad").cast("decimal(18,2)") *
         F.col("precio_unitario").cast("decimal(18,2)")).cast("decimal(18,2)")
    )
    .withColumn(
        "importe_descuento",
        (F.col("importe_bruto") *
         (F.col("descuento_pct").cast("decimal(8,4)") / F.lit(100))).cast("decimal(18,2)")
    )
    .withColumn(
        "importe_neto",
        (F.col("importe_bruto") - F.col("importe_descuento")).cast("decimal(18,2)")
    )
    .withColumn(
        "costo_estimado",
        (F.col("cantidad").cast("decimal(18,2)") *
         F.col("costo_unitario_referencia").cast("decimal(18,2)")).cast("decimal(18,2)")
    )
    .withColumn(
        "utilidad_bruta_estimada",
        (F.col("importe_neto") - F.col("costo_estimado")).cast("decimal(18,2)")
    )
    .select(
        "venta_id",
        "fecha_key", "cliente_key", "producto_key", "almacen_key",
        "fecha_venta", "cantidad", "precio_unitario", "descuento_pct",
        "importe_bruto", "importe_descuento", "importe_neto",
        "costo_estimado", "utilidad_bruta_estimada",
        "metodo_pago", "_source_file", "_ingestion_timestamp"
    )
)

print("Ventas Silver:", conteo_ventas_silver)
print("Ventas mapeadas para el hecho:", df_fact_ventas.count())
print("Ventas sin mapeo dimensional:", df_ventas_no_mapeadas.count())
print("Total conciliado:", df_fact_ventas.count() + df_ventas_no_mapeadas.count())

## 7. Inspección de `fact_ventas`

**¿Qué realiza esta sección?** Muestra una muestra de las ventas enriquecidas con claves sustitutas y medidas calculadas. Permite revisar que los importes sean coherentes antes de persistir.

In [ ]:
print("Registros candidatos para fact_ventas:", df_fact_ventas.count())
df_fact_ventas.printSchema()
display(df_fact_ventas.orderBy("venta_id").limit(20))

## 8. Conciliación y controles previos a la escritura

**¿Qué realiza esta sección?** Comprueba que ninguna venta se haya perdido durante el mapeo, que `venta_id` siga siendo único y que las claves dimensionales del hecho estén completas.

Las ventas no mapeadas se contabilizan aparte y no se incorporan silenciosamente al hecho.

In [ ]:
conteo_fact = df_fact_ventas.count()
conteo_no_mapeadas = df_ventas_no_mapeadas.count()

assert conteo_ventas_silver == conteo_fact + conteo_no_mapeadas, \
    "La conciliación entre Silver y ventas mapeadas/no mapeadas no coincide."

assert df_fact_ventas.groupBy("venta_id").count().filter(F.col("count") > 1).count() == 0, \
    "Hay venta_id duplicados en fact_ventas."

for clave in ["fecha_key", "cliente_key", "producto_key", "almacen_key"]:
    assert df_fact_ventas.filter(F.col(clave).isNull()).count() == 0, \
        f"Hay claves nulas en {clave}."

print("Ventas Silver:", conteo_ventas_silver)
print("Fact ventas:", conteo_fact)
print("Ventas no mapeadas:", conteo_no_mapeadas)
print("Total conciliado:", conteo_fact + conteo_no_mapeadas)
print("Conciliación y controles previos correctos.")

if conteo_no_mapeadas > 0:
    display(
        df_ventas_no_mapeadas
        .groupBy("motivo_no_mapeo")
        .count()
        .orderBy("motivo_no_mapeo")
    )

## 9. Preparar esquema y persistir tablas Delta

**¿Qué realiza esta sección?** Crea el esquema `golden` si no existe y persiste `fact_ventas`. También persiste la tabla de ventas no mapeadas para conservar trazabilidad y conciliación.

La escritura utiliza `overwrite`: si se vuelve a ejecutar, reconstruye estas dos tablas con los datos actuales de Silver y dimensiones. No modifica Bronze, Silver ni las dimensiones.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaGolden}")

(
    df_fact_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}fact_ventas/")
    .saveAsTable(tabla_fact_ventas)
)
print("Tabla creada:", tabla_fact_ventas)

(
    df_ventas_no_mapeadas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}fact_ventas_no_mapeadas/")
    .saveAsTable(tabla_fact_ventas_no_mapeadas)
)
print("Tabla creada:", tabla_fact_ventas_no_mapeadas)

## 10. Lectura de tablas persistidas

**¿Qué realiza esta sección?** Vuelve a leer las dos tablas desde Unity Catalog para verificar que la persistencia Delta se haya completado correctamente.

In [ ]:
df_fact_ventas_tabla = spark.table(tabla_fact_ventas)
df_ventas_no_mapeadas_tabla = spark.table(tabla_fact_ventas_no_mapeadas)

print("Fact ventas persistidas:", df_fact_ventas_tabla.count())
print("Ventas no mapeadas persistidas:", df_ventas_no_mapeadas_tabla.count())

## 11. Validación final de `fact_ventas`

**¿Qué realiza esta sección?** Ejecuta los controles finales sobre las tablas persistidas: conciliación, unicidad de `venta_id`, integridad de claves dimensionales y ausencia de valores nulos en los indicadores calculados.

La carga se considera conciliada cuando las ventas de Silver equivalen a las filas de `fact_ventas` más las ventas conservadas en `fact_ventas_no_mapeadas`.

In [ ]:
conteo_fact_final = df_fact_ventas_tabla.count()
conteo_no_mapeadas_final = df_ventas_no_mapeadas_tabla.count()

assert conteo_ventas_silver == conteo_fact_final + conteo_no_mapeadas_final, \
    "La conciliación final no coincide con Silver."

assert df_fact_ventas_tabla.groupBy("venta_id").count().filter(F.col("count") > 1).count() == 0, \
    "Se encontraron venta_id duplicados en la tabla persistida."

for clave in ["fecha_key", "cliente_key", "producto_key", "almacen_key"]:
    assert df_fact_ventas_tabla.filter(F.col(clave).isNull()).count() == 0, \
        f"Se encontraron claves nulas en {clave}."

for medida in [
    "importe_bruto", "importe_descuento", "importe_neto",
    "costo_estimado", "utilidad_bruta_estimada"
]:
    assert df_fact_ventas_tabla.filter(F.col(medida).isNull()).count() == 0, \
        f"Se encontraron valores nulos en {medida}."

print("Ventas Silver:", conteo_ventas_silver)
print("Fact ventas:", conteo_fact_final)
print("Ventas no mapeadas:", conteo_no_mapeadas_final)
print("Total conciliado:", conteo_fact_final + conteo_no_mapeadas_final)
print("Validación final de fact_ventas correcta.")

display(df_fact_ventas_tabla.orderBy("venta_id").limit(10))